# Phase 03 — Psychological Capital → Employability

## Step 13 — Final Accuracy Improvement Experiments

This notebook documents the Phase 03 accuracy improvement run using the expanded unprepared dataset `data/raw/phase03_psycap_new.csv`.

It repeats the Phase 03 evidence order from notebooks 01–08 and 11:

1. preprocess, rename items, and validate Likert scores
2. reliability
3. construct scores
4. correlation
5. feature sets A (constructs) / B (item-level research) / C (overall PsyCap research)
6. repeated CV model comparison
7. nested CV confirmation
8. freeze only if improved

Notebooks 01–12 remain the original 1000-row development evidence. Phase 01 and Phase 02 files are not modified.

### Previous frozen model
- ElasticNet + StandardScaler on 4 PsyCap constructs
- Nested CV mean R² = **0.574** (1000 rows)

### New dataset
- `phase03_psycap_new.csv`: **1500 rows** (266 real, 1234 synthetic)
- Unprepared questionnaire columns renamed with the notebook 01 codebook
- No exact duplicate rows removed

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "phase3_psycap":
    PROJECT_DIR = PROJECT_DIR.parent.parent

RAW_FILE = PROJECT_DIR / "data" / "raw" / "phase03_psycap_new.csv"
PREPARED = PROJECT_DIR / "data" / "processed" / "phase3_psycap_prepared.csv"
ACCURACY_DIR = PROJECT_DIR / "reports" / "phase3" / "accuracy_improvement"
MODEL_PATH = PROJECT_DIR / "models" / "phase3" / "phase3_psycap_selected_model.pkl"

print("RAW exists:", RAW_FILE.exists())
print("Prepared exists:", PREPARED.exists())
print("Model exists:", MODEL_PATH.exists())

## 1. Preprocessing summary

Notebook 01 rules applied to the new file:
- rename full question text to `PsyCap1`–`PsyCap12` and `EVS1`–`EVS12`
- keep Likert values in 1–5
- remove exact duplicate rows (0 removed)
- compute Hope, Optimism, Resilience, Self-Efficacy and `EVS_avg`

Saved outputs:
- `data/processed/phase3_psycap_prepared.csv`
- `data/processed/phase3_psycap_construct_features.csv`
- `data/processed/phase3_psycap_item_features.csv`
- `data/processed/phase3_psycap_overall_feature.csv`

In [ ]:
prepared = pd.read_csv(PREPARED)
print("Rows:", len(prepared))
print(prepared["data_source"].value_counts())
prepared[["Hope_avg", "Optimism_avg", "Resilience_avg", "SelfEfficacy_avg", "EVS_avg"]].describe().round(3)

## 2. Repeated CV model comparison

Feature Set A (deployable): `Hope_avg`, `Optimism_avg`, `Resilience_avg`, `SelfEfficacy_avg`

Feature Set B (research only): `PsyCap1`–`PsyCap12`

Feature Set C (research only): `PsyCap_avg`

In [ ]:
comparison = pd.read_csv(ACCURACY_DIR / "Model_Comparison_RepeatedCV.csv")
comparison[comparison["Deployable"] == True].head(10)

## 3. Nested CV confirmation

Top deployable candidates confirmed with 5-fold nested CV on Feature Set A.

In [ ]:
nested = pd.read_csv(ACCURACY_DIR / "Final_Candidate_Nested_Comparison.csv")
nested

## 4. Final replacement decision

**ElasticNet retained** because nested OOF R² (0.933) was within 0.005 of Huber (0.934), preserving the Phase 03 interpretability and SHAP evidence architecture from notebooks 09–12.

| Metric | Previous (1000 rows) | New (1500 rows) |
|--------|---------------------|-----------------|
| Nested OOF R² | 0.574 | **0.933** |
| Test R² | 0.399 (old Step 07 hold-out) | **0.918** |
| Nested MAE | 0.136 | **0.167** |
| Nested RMSE | 0.230 | **0.246** |

In [ ]:
decision = pd.read_csv(ACCURACY_DIR / "Final_Model_Replacement_Decision.csv")
decision

## 5. Frozen model artifacts

Updated files:
- `models/phase3/phase3_psycap_selected_model.pkl`
- `models/phase3/phase3_psycap_model.pkl`
- `models/phase3/phase3_psycap_model_metadata.json`
- `models/phase3/phase3_psycap_freeze_manifest.json`
- `models/phase3/phase3_psycap_feature_schema.json`
- `reports/phase3/final_validation/Reference_Predictions.csv`
- `reports/phase3/final_validation/Final_Model_Metrics.csv`

In [ ]:
import json
import joblib

metadata = json.loads((PROJECT_DIR / "models" / "phase3" / "phase3_psycap_model_metadata.json").read_text())
print(json.dumps(metadata["metrics"], indent=2))

model = joblib.load(MODEL_PATH)
sample = prepared[["Hope_avg", "Optimism_avg", "Resilience_avg", "SelfEfficacy_avg"]].head(3)
print("Sample predictions:", model.predict(sample).round(3))